# Week 4 Wednesday — Regional and Delivery-Stage Analysis
**Owner:** Jaykishan (Member 3)

This notebook reads the Wednesday SQL exports, validates their columns and calculations, and creates charts. Run the SQL queries first and replace the header-only CSV templates with actual MySQL Workbench exports. The templates contain no fabricated results.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

EXPORT_DIR = Path("../data/exports")
regional_path = EXPORT_DIR / "week4_regional_delay_analysis.csv"
state_path = EXPORT_DIR / "week4_state_delay_analysis.csv"
stage_path = EXPORT_DIR / "week4_delivery_stage_analysis.csv"


## 1. Load exported SQL results
The CSV files in the starter package are header-only templates. If they are still empty, export the actual query results from MySQL Workbench before continuing.

In [ ]:
def load_export(path):
    df = pd.read_csv(path)
    if df.empty:
        raise ValueError(
            f"{path.name} has headers but no result rows. "
            "Run the matching SQL query and export its result grid first."
        )
    return df

regional = load_export(regional_path)
state = load_export(state_path)
stage = load_export(stage_path)

display(regional.head())
display(state.head())
display(stage.head())


## 2. Validate regional and state data

In [ ]:
required_regional = {
    "region", "total_orders", "eligible_delivered_orders",
    "late_orders", "on_time_orders", "late_delivery_rate_pct"
}
required_state = {
    "customer_state", "total_orders", "eligible_delivered_orders",
    "late_orders", "on_time_orders", "late_delivery_rate_pct"
}
assert required_regional.issubset(regional.columns), "Regional export is missing expected columns."
assert required_state.issubset(state.columns), "State export is missing expected columns."

for name, df in [("regional", regional), ("state", state)]:
    print(f"\\n{name.upper()} missing values:")
    display(df.isna().sum().to_frame("missing_values"))
    print(f"{name} rows:", len(df))


## 3. Recalculate late-delivery rates

In [ ]:
for label, df in [("Regional", regional), ("State", state)]:
    recalculated = (
        df["late_orders"] / df["eligible_delivered_orders"].replace(0, np.nan) * 100
    ).round(2)
    comparison = pd.DataFrame({
        "reported_rate_pct": df["late_delivery_rate_pct"],
        "recalculated_rate_pct": recalculated,
    })
    comparison["matches_rounded"] = (
        comparison["reported_rate_pct"].round(2) == comparison["recalculated_rate_pct"]
    )
    print(label, "rate validation:")
    display(comparison)


## 4. Regional late-delivery chart

In [ ]:
regional_plot = regional.sort_values("late_delivery_rate_pct", ascending=False)
plt.figure(figsize=(10, 6))
plt.bar(regional_plot["region"].astype(str), regional_plot["late_delivery_rate_pct"])
plt.title("Late Delivery Rate by Customer Region")
plt.xlabel("Customer region")
plt.ylabel("Late delivery rate (%)")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.savefig("../powerbi/screenshots/regional_analysis.png", dpi=160, bbox_inches="tight")
plt.show()


## 5. State-level late-delivery chart

In [ ]:
state_plot = state.sort_values("late_delivery_rate_pct", ascending=False)
plt.figure(figsize=(12, 6))
plt.bar(state_plot["customer_state"].astype(str), state_plot["late_delivery_rate_pct"])
plt.title("Late Delivery Rate by Customer State")
plt.xlabel("Customer state")
plt.ylabel("Late delivery rate (%)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig("../powerbi/screenshots/state_analysis.png", dpi=160, bbox_inches="tight")
plt.show()


## 6. Delivery-stage duration chart

In [ ]:
stage_row = stage.iloc[0]
stage_labels = [
    "Purchase to approval",
    "Approval to handoff",
    "Handoff to delivery",
]
stage_values = [
    stage_row["avg_purchase_to_approval_days"],
    stage_row["avg_approval_to_handoff_days"],
    stage_row["avg_handoff_to_delivery_days"],
]
plt.figure(figsize=(10, 6))
plt.bar(stage_labels, stage_values)
plt.title("Average Duration by Delivery Stage")
plt.xlabel("Delivery stage")
plt.ylabel("Average duration (days)")
plt.xticks(rotation=15, ha="right")
plt.tight_layout()
plt.savefig("../powerbi/screenshots/delivery_stage_analysis.png", dpi=160, bbox_inches="tight")
plt.show()


## 7. Evidence-based observations

In [ ]:
highest_rate_region = regional.loc[regional["late_delivery_rate_pct"].idxmax()]
highest_volume_region = regional.loc[regional["total_orders"].idxmax()]
print("Highest regional late-delivery rate:", highest_rate_region["region"],
      f'({highest_rate_region["late_delivery_rate_pct"]}%; '
      f'{highest_rate_region["eligible_delivered_orders"]} eligible orders)')
print("Highest regional order volume:", highest_volume_region["region"],
      f'({highest_volume_region["total_orders"]} orders)')
print("Delivery-stage averages (days):")
display(pd.DataFrame({"stage": stage_labels, "average_days": stage_values}))
print("Negative interval counts:")
for col in [
    "negative_purchase_to_approval_stages",
    "negative_approval_to_handoff_stages",
    "negative_handoff_to_delivery_stages",
]:
    if col in stage_row.index:
        print(col, int(stage_row[col]))


## 8. Interpretation and limitations
- Region is derived from customer state; it is not a carrier operating region.
- State and regional rates must always be interpreted alongside eligible delivered-order counts.
- These results are descriptive and do not establish causes of delays.
- Negative timestamp intervals are quality flags and require investigation; they are not silently removed from the reported counts.
- A carrier handoff timestamp does not identify the carrier. Do not substitute `seller_id` for a carrier identifier.
- Only write findings based on the actual SQL exports loaded above.
